Since the function is the norm squared of a vector difference, we can express is as, assuming the 2-norm
$$
E[\sum_{i=1}^n (X_i - \theta_i)^2]
$$
if we expand we get
$$
E[\sum_{i=1}^n X_i^2 + \theta_i^2 - 2X_i\theta_i] = \sum_{i=1}^n E[ X_i^2] + \sum_{i=1}^n[\theta_i^2] - 2\sum_{i=1}^n E[X_i]\theta_i
$$
Which is zero if $\theta_i = E[X_i]$

We consider the problem of minimizing
$$
f(\theta) = \mathbb{E}\left[\|X-\theta\|^2\right],
$$
where $X=(X_1,\ldots,X_{100})$ has independent coordinates
$$
X_i \sim \mathcal{N}(i,1).
$$

The theoretical minimizer is the vector of coordinate-wise expectations:
$$
\theta^* = \mathbb{E}[X] = (1,2,\ldots,100).
$$

Since the objective is quadratic, its gradient is
$$
\nabla f(\theta) = 2(\theta-\mathbb{E}[X]).
$$

Therefore, finding the minimum is equivalent to finding the root of the gradient:
$$
\nabla f(\theta)=0.
$$

The Robbins-Monro algorithm can be used to find this root by recursively updating
$$
\theta_{n+1}
=
\theta_n-a_n G(\theta_n,X_{n+1}),
$$
where $G(\theta_n,X_{n+1})$ is an unbiased estimate of the gradient. In our case,
$$
G(\theta_n,X_{n+1})=2(\theta_n-X_{n+1}).
$$

The Kiefer-Wolfowitz algorithm, on the other hand, estimates the gradient using finite differences:
$$
\widehat{\nabla_i f(\theta)}
=
\frac{
Z(X,\theta+\frac{h_n}{2}e_i)
-
Z(\widetilde X,\theta-\frac{h_n}{2}e_i)
}{h_n},
$$
where $X$ and $\widetilde X$ are independent samples and $e_i$ is the $i$-th standard basis vector.

The finite-difference estimator introduces considerably more noise than the direct gradient estimator used by Robbins-Monro. In particular, when estimating a single partial derivative, the difference between the two objective evaluations contains random contributions from all 100 coordinates. Furthermore, dividing by a small perturbation parameter $h_n$ amplifies this noise.

At the theoretical minimizer, the variance of the finite-difference estimator is
$$
\operatorname{Var}(\widehat{\nabla_i f(\theta^*)})
=
\frac{400}{h_n^2}+2.
$$

In contrast, the variance of the Robbins-Monro gradient estimator is
$$
\operatorname{Var}(G_i(\theta^*,X))=4.
$$

This explains why Robbins-Monro performs substantially better than Kiefer-Wolfowitz in this high-dimensional example.

Overall, the experiment illustrates the disadvantage of estimating gradients through finite differences in high dimensions. Since we know that the root of the gradient corresponds to the global minimum for our quadratic objective, Robbins-Monro can exploit direct stochastic gradient estimates, whereas Kiefer-Wolfowitz must approximate the gradient numerically, resulting in substantially higher variance and slower convergence.

In [1]:
from week10 import *

# Define the function
z = lambda x, theta : np.linalg.norm(x-theta)**2

# Sample from X_i ~N(i,1)
N = 100
mus = np.arange(1, N + 1)
sampler = lambda : (
    np.random.normal(mus, 1),
    np.random.normal(mus, 1),
)
# Step sizes
a = [0.5/n for n in range(1, 10002)]
h = [2] * 10001
theta0 = np.zeros(N)
keifer_wolfowitz(z, sampler, theta0, h, a, max_iter= 5000)

Maximum iterations reached without convergence


array([ 0.92818492,  1.91368886,  2.92146009,  3.92603083,  4.93737184,
        5.94279238,  6.94305065,  7.92019844,  8.91202422,  9.92179244,
       10.93594492, 11.92221512, 12.90496785, 13.92410109, 14.93014409,
       15.92096926, 16.92760264, 17.92812751, 18.92631747, 19.92627849,
       20.93252386, 21.93424298, 22.93043079, 23.93664634, 24.9272858 ,
       25.94373924, 26.92433995, 27.91037181, 28.9386072 , 29.93507147,
       30.92606554, 31.92176861, 32.93123768, 33.93098768, 34.9257805 ,
       35.92871342, 36.92691643, 37.93298532, 38.9229254 , 39.91207715,
       40.92977379, 41.92239982, 42.92193598, 43.93648056, 44.92891918,
       45.92678652, 46.90719164, 47.94107958, 48.92839351, 49.92558466,
       50.92185169, 51.90752374, 52.92280155, 53.93488   , 54.92463395,
       55.92814658, 56.9105163 , 57.91213229, 58.93282344, 59.92346833,
       60.93181379, 61.92976153, 62.9191956 , 63.92283702, 64.92308835,
       65.96141185, 66.90813857, 67.9184958 , 68.92505398, 69.91

We now use SGD instead, we fint the root of the gradient being 0.

In [9]:
dz = lambda x, theta : 2 * (x - theta)
sampler = lambda : np.random.normal(mus, 1)
a = [0.001/n**0.7 for n in range(1, 10002)]
robbins_monro(dz , sampler, theta0, a, max_iter= 5000)

Convergence at iteration 193


array([ 0.92617265,  1.91167659,  2.91944781,  3.92401856,  4.93535956,
        5.94078011,  6.94103837,  7.91818616,  8.91001194,  9.91978017,
       10.93393264, 11.92020284, 12.90295558, 13.92208881, 14.92813181,
       15.91895699, 16.92559037, 17.92611524, 18.92430519, 19.92426622,
       20.93051159, 21.93223071, 22.92841851, 23.93463407, 24.92527352,
       25.94172697, 26.92232768, 27.90835954, 28.93659493, 29.9330592 ,
       30.92405327, 31.91975634, 32.9292254 , 33.92897541, 34.92376822,
       35.92670115, 36.92490416, 37.93097304, 38.92091313, 39.91006488,
       40.92776152, 41.92038755, 42.91992371, 43.93446828, 44.92690691,
       45.92477424, 46.90517937, 47.93906731, 48.92638124, 49.92357239,
       50.91983942, 51.90551146, 52.92078927, 53.93286773, 54.92262168,
       55.92613431, 56.90850402, 57.91012002, 58.93081117, 59.92145606,
       60.92980152, 61.92774926, 62.91718333, 63.92082475, 64.92107608,
       65.95939958, 66.9061263 , 67.91648352, 68.92304171, 69.91